# Bronze Layer - Azure ADLS Gen2 Ingestion

Ingesta de datos raw desde Azure ADLS Gen2 usando la Storage Python SDK.
Compatible con Databricks Serverless (no requiere cluster clásico).

## Flujo
1. Instalar SDK de Azure Storage
2. Configurar credenciales del Service Principal
3. Listar y leer archivos del contenedor
4. Guardar como tabla Delta en bronze.raw

In [0]:
%pip install azure-storage-file-datalake azure-identity --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from azure.storage.filedatalake import DataLakeServiceClient
from azure.identity import ClientSecretCredential

SCOPE = "eurobank-secrets"
AZURE_TENANT_ID = dbutils.secrets.get(scope=SCOPE, key="azure-tenant-id")
AZURE_CLIENT_ID = dbutils.secrets.get(scope=SCOPE, key="azure-client-id")
AZURE_CLIENT_SECRET = dbutils.secrets.get(scope=SCOPE, key="azure-client-secret")
AZURE_STORAGE_ACCOUNT = "stkeurobankoyvixj"
AZURE_CONTAINER = "eurobank-logs"

credential = ClientSecretCredential(
    tenant_id=AZURE_TENANT_ID,
    client_id=AZURE_CLIENT_ID,
    client_secret=AZURE_CLIENT_SECRET
)

account_url = f"https://{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net"
service_client = DataLakeServiceClient(account_url=account_url, credential=credential)
file_system_client = service_client.get_file_system_client(file_system=AZURE_CONTAINER)

print(f"Conectado a: {AZURE_STORAGE_ACCOUNT}")

Conectado a: stkeurobankoyvixj


In [0]:
paths = file_system_client.get_paths(path="mobile/")

for path in paths:
    if not path.is_directory:
        print(f"{path.name} ({path.content_length} bytes)")

mobile/mobile_logs_batch_001.json.gz (5364 bytes)


In [0]:
import gzip
import json
import pandas as pd

def read_json_gz(file_system_client, file_path):
    file_client = file_system_client.get_file_client(file_path)
    download = file_client.download_file()
    compressed = download.readall()
    decompressed = gzip.decompress(compressed)
    content = decompressed.decode('utf-8')

    data = []
    for line in content.strip().split('\n'):
        if line.strip():
            record = json.loads(line)
            if 'metadata' in record and record['metadata'] is not None and not isinstance(record['metadata'], str):
                record['metadata'] = json.dumps(record['metadata'])
            data.append(record)
    return pd.DataFrame(data)

paths = file_system_client.get_paths(path="mobile/")
frames = []

for path in paths:
    if path.name.endswith('.json.gz') and not path.is_directory:
        df = read_json_gz(file_system_client, path.name)
        frames.append(df)
        print(f"{path.name}: {len(df)} registros")

if frames:
    combined = pd.concat(frames, ignore_index=True)
    spark_df = spark.createDataFrame(combined)

    table_name = "eurobank_bronze.raw.bronze_mobile_logs"
    spark_df.write \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .saveAsTable(table_name)
    print(f"Guardado en {table_name}: {spark_df.count()} registros")
    display(spark_df.limit(5))
else:
    print("No se encontraron archivos .json.gz en mobile/")

mobile/mobile_logs_batch_001.json.gz: 100 registros
Guardado en eurobank_bronze.raw.bronze_mobile_logs: 100 registros


event_id,timestamp,user_id,user_email,event_type,device_model,os_version,app_version,session_id,ip_address,metadata
80973e32-ad64-42c9-b971-cbc427e46e64,2026-09-10T16:19:57.634939+00:00,usr-1004,marco.rossi@eurobank.com,LOGOUT,iPhone 15,Android 14,v3.11.4,d05eae36,192.168.85.197,"{""screen"": ""SecurityView"", ""latency_ms"": 57, ""is_jailbroken_or_rooted"": false}"
f96a2fef-5cbc-4409-a6de-39c1792d5227,2026-09-10T16:19:57.635163+00:00,usr-1001,carlos.mendoza@eurobank.com,LOGOUT,iPhone 16 Pro,iOS 17.5,v3.12.1,b7a1d3f9,192.168.31.49,"{""screen"": ""SecurityView"", ""latency_ms"": 318, ""is_jailbroken_or_rooted"": false}"
fc8317c4-c70a-4e6b-8518-6106abfd660a,2026-09-10T16:19:57.635172+00:00,usr-1003,elena.dupont@eurobank.com,LOGIN_SUCCESS,Pixel 9 Pro,Android 14,v3.12.1,61bf5f53,192.168.109.117,"{""screen"": ""SecurityView"", ""latency_ms"": 266, ""is_jailbroken_or_rooted"": false}"
d3cf9001-739a-4d39-be79-6153a93e5b0c,2026-09-10T16:19:57.635179+00:00,usr-1001,carlos.mendoza@eurobank.com,LOGIN_FAILED,iPhone 16 Pro,Android 15,v3.12.1,f216859d,192.168.204.209,"{""screen"": ""SecurityView"", ""latency_ms"": 98, ""is_jailbroken_or_rooted"": false}"
38633aab-58d4-4b16-aba1-0e0e4f3d9c89,2026-09-10T16:19:57.635184+00:00,usr-1001,carlos.mendoza@eurobank.com,CARD_BLOCKED,iPhone 16 Pro,Android 15,v3.12.1,f3e22228,192.168.198.36,"{""screen"": ""SecurityView"", ""latency_ms"": 101, ""is_jailbroken_or_rooted"": false}"
